# Task 3.6: Model Improvement - Iteration 4

**Project:** Potato Quality Classification (Fresh vs. Rotten)  
**ILO:** 8.2 D  

This notebook improves upon Iteration 3 (MobileNetV2, frozen, 128x128) by increasing the input image resolution to **224x224 pixels**, the native ImageNet resolution that MobileNetV2 was originally trained on.

### Why change the image resolution?
In all previous iterations, images were resized to 128x128 pixels. However, MobileNetV2 was pre-trained on ImageNet at 224x224 resolution. By using the native resolution:
- The pre-trained filters operate on the resolution they were designed for
- More spatial detail is preserved (important for detecting subtle rot patterns)
- The feature maps contain richer information for the classifier head

| | Iteration 3 | Iteration 4 |
|---|---|---|
| Base model | MobileNetV2 (frozen) | MobileNetV2 (frozen) |
| Input resolution | 128 x 128 | 224 x 224 |
| Augmentation | None | None |
| Classification head | Same | Same |

Everything else remains identical to isolate the effect of resolution.

---

**Results so far:**
| Model | Accuracy |
|---|---|
| Random Guess | 50.18% |
| MLP Baseline | 79.03% |
| CNN v1 (basic CNN, 128x128) | 94.35% |
| CNN v2 (+ augmentation, 128x128) | 93.28% |
| CNN v3 (MobileNetV2 frozen, 128x128) | 95.43% |

---
## 1. Imports & Setup

In [ ]:
!pip install seaborn scikit-learn -q

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

---
## 2. Data Loading & Preprocessing

The key change: **IMG_SIZE is now 224x224** instead of 128x128. This is the native resolution MobileNetV2 was trained on. No augmentation is applied, same as Iteration 3.

In [ ]:
# -- Paths --
BASE_PATH  = 'data/processed'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VAL_PATH   = os.path.join(BASE_PATH, 'val')
TEST_PATH  = os.path.join(BASE_PATH, 'test')

IMG_SIZE   = (224, 224)   # Changed from 128x128 to 224x224 (native ImageNet resolution)
BATCH_SIZE = 32

# -- Remove .ipynb_checkpoints if present --
import shutil, glob
for cp in glob.glob(os.path.join(BASE_PATH, '*', '.ipynb_checkpoints')):
    shutil.rmtree(cp)
    print(f'Removed: {cp}')

# -- No augmentation: only rescaling --
datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_gen = datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

CLASS_NAMES = list(train_gen.class_indices.keys())

# -- Verify --
assert len(train_gen.class_indices) == 2, f"ERROR: Expected 2 classes, got {train_gen.class_indices}"
print('Classes      :', train_gen.class_indices)
print('Train samples:', train_gen.samples)
print('Val samples  :', val_gen.samples)
print('Test samples :', test_gen.samples)
print(f'Train batches: {len(train_gen)}')
print(f'Image size   : {IMG_SIZE}')
print('\n✅ Data loaded correctly - 2 classes found.')

---
## 3. CNN Architecture - cnn_v4 (MobileNetV2 @ 224x224)

Same architecture as Iteration 3, but the input shape is now 224x224x3 instead of 128x128x3. The pre-trained MobileNetV2 base is fully frozen.

In [ ]:
# -- Load pre-trained MobileNetV2 at native resolution --
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)   # Native ImageNet resolution
)

# -- Freeze all layers --
base_model.trainable = False

print(f'Base model: {base_model.name}')
print(f'Total layers: {len(base_model.layers)}')
print(f'Trainable: {sum(1 for l in base_model.layers if l.trainable)}')
print(f'Frozen: {sum(1 for l in base_model.layers if not l.trainable)}')

In [ ]:
# -- Build full model --
cnn_v4 = Sequential([
    base_model,

    # -- Classification Head (same as Iteration 3) --
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(2, activation='softmax')
], name='cnn_v4_mobilenet_224')

cnn_v4.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_v4.summary()

---
## 4. Training with Early Stopping

Same training configuration as Iteration 3. Training will be slightly slower per epoch because 224x224 images have 3x more pixels than 128x128.

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history_v4 = cnn_v4.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen,
    callbacks=[early_stop],
    verbose=1
)

print(f'\nTraining stopped after {len(history_v4.history["loss"])} epoch(s).')

---
## 5. Learning Curves

In [ ]:
def plot_learning_curves(history, title_suffix=''):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'],     label='Train loss')
    axes[0].plot(history.history['val_loss'], label='Val loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Loss {title_suffix}')
    axes[0].legend()

    axes[1].plot(history.history['accuracy'],     label='Train accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Val accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'Accuracy {title_suffix}')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_learning_curves(history_v4, title_suffix='- CNN v4 (MobileNetV2 @ 224x224)')

---
## 6. Evaluation on the Test Set

In [ ]:
# -- Predict on test set --
test_gen.reset()
y_pred_proba = cnn_v4.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_gen.classes

# -- Metrics --
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted')
rec  = recall_score(y_true, y_pred, average='weighted')
f1   = f1_score(y_true, y_pred, average='weighted')

print('=' * 45)
print(f'  Accuracy  : {acc  * 100:.2f} %')
print(f'  Precision : {prec * 100:.2f} %')
print(f'  Recall    : {rec  * 100:.2f} %')
print(f'  F1 Score  : {f1   * 100:.2f} %')
print('=' * 45)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# -- Confusion matrix heatmap --
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - CNN v4 (Test Set)')
plt.tight_layout()
plt.show()

---
## 7. Comparison: All Iterations

In [ ]:
RANDOM_GUESS_ACC = 50.18
MLP_BASELINE_ACC = 79.03
CNN_V1_ACC       = 94.35
CNN_V2_ACC       = 93.28
CNN_V3_ACC       = 95.43   # MobileNetV2 @ 128x128
cnn_v4_acc       = acc * 100

models   = ['Random\nGuess', 'MLP\nBaseline', 'CNN v1\n(basic\n128x128)', 'CNN v2\n(aug\n128x128)', 'CNN v3\n(MobileNet\n128x128)', 'CNN v4\n(MobileNet\n224x224)']
acc_vals = [RANDOM_GUESS_ACC, MLP_BASELINE_ACC, CNN_V1_ACC, CNN_V2_ACC, CNN_V3_ACC, cnn_v4_acc]
colours  = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd', '#e377c2']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(models, acc_vals, color=colours, edgecolor='black', width=0.5)

for bar, val in zip(bars, acc_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.2f}%',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison - All Iterations', fontsize=13)
plt.tight_layout()
plt.show()

print('\n-- Comparison Table --')
print(f'  Random Guess              : {RANDOM_GUESS_ACC:.2f}%')
print(f'  MLP Baseline              : {MLP_BASELINE_ACC:.2f}%')
print(f'  CNN v1 (basic, 128x128)   : {CNN_V1_ACC:.2f}%')
print(f'  CNN v2 (aug, 128x128)     : {CNN_V2_ACC:.2f}%')
print(f'  CNN v3 (MobileNet, 128x128): {CNN_V3_ACC:.2f}%')
print(f'  CNN v4 (MobileNet, 224x224): {cnn_v4_acc:.2f}%')
delta = cnn_v4_acc - CNN_V3_ACC
print(f'\n  Effect of higher resolution: {delta:+.2f} percentage points')

In [ ]:
# -- Save the best model for use in Error Analysis --
import os
save_path = os.path.join('data', 'cnn_v4_best.keras')
cnn_v4.save(save_path)
print(f'Model saved to: {os.path.abspath(save_path)}')

---
## 8. Summary & Conclusions Iteration 4

**What changed from Iteration 3:** The input image resolution was increased from 128x128 to 224x224 pixels, which is the native resolution MobileNetV2 was pre-trained on. Everything else remained identical: frozen MobileNetV2 base, same classification head, no augmentation, same optimizer.

**Why 224x224:** MobileNetV2 was originally trained on ImageNet at 224x224 resolution. Using this native resolution means the pre-trained convolutional filters operate on the scale they were designed for, producing more accurate feature representations. Additionally, the higher resolution preserves more spatial detail, which is important for detecting subtle signs of decay in potatoes.

**Results:**
- CNN v3 (MobileNetV2, 128x128): 95.43%
- CNN v4 (MobileNetV2, 224x224): 97.32%
- Effect: +1.89 percentage points

**Analysis:** Increasing the input resolution from 128x128 to 224x224 provided a significant improvement, confirming that MobileNetV2 performs best at its native ImageNet resolution. The model achieved 97.32% test accuracy with balanced precision and recall (both 97%) across both classes. The learning curves show stable training with minimal overfitting, indicating that the frozen feature extraction approach combined with the correct input resolution is highly effective for this dataset.

**Overall best model:** Iteration 4 (MobileNetV2 frozen, 224x224) is the best performing model with 97.32% test accuracy. The combination of transfer learning with a frozen pre-trained base at native resolution proved to be the most effective approach for potato quality classification on this dataset of 3,712 images.